# Machine Learning Data Preprocessing: A Comprehensive Guide

Data preprocessing is a crucial first step in any machine learning pipeline. Raw real-world data is often incomplete, noisy, inconsistent, and improperly formatted for machine learning algorithms. Effective preprocessing improves model accuracy, training speed, and generalization capability.

### Table of Contents:
1. **Environment Setup & Library Imports**
2. **Loading & Preparing Sample Dataset**
3. **Exploratory Data Analysis (EDA)**
4. **Handling Missing Values (Imputation)**
5. **Encoding Categorical Variables**
6. **Feature Engineering**
7. **Train-Test Split & Data Leakage Prevention**
8. **Feature Scaling**
9. **Feature Selection Techniques**
10. **Summary & Conclusion**

## 1. Environment Setup & Library Imports

In this section, we import the core Python libraries required for data manipulation, visualization, numerical operations, and machine learning preprocessing.

In [ ]:
# Installation commands for required packages (uncomment if missing in your environment)
# !pip install numpy pandas scikit-learn matplotlib seaborn

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
try:
    from IPython.display import display
except ImportError:
    display = print

# Scikit-Learn Imports
from sklearn.datasets import load_breast_cancer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import (
    LabelEncoder,
    OneHotEncoder,
    OrdinalEncoder,
    StandardScaler,
    MinMaxScaler,
    RobustScaler,
    PolynomialFeatures
)
from sklearn.feature_selection import SelectKBest, f_classif, RFE
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

# Set random seed for reproducibility
np.random.seed(42)

# Configure plotting aesthetics
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

print("All libraries successfully imported!")

## 2. Loading & Preparing a Sample Dataset

To demonstrate all data preprocessing steps realistically (including missing value handling, numerical processing, and categorical encoding), we load Scikit-Learn's built-in **Breast Cancer Wisconsin Dataset** and enrich it with synthetic categorical columns and missing values (`NaN`s).

In [ ]:
# Load built-in dataset from sklearn
cancer_data = load_breast_cancer(as_frame=True)
df_raw = cancer_data.frame.copy()

# Select a subset of features for clean visualization and processing
selected_features = [
    'mean radius', 'mean texture', 'mean perimeter', 'mean area',
    'mean smoothness', 'mean compactness', 'mean concavity'
]
df = df_raw[selected_features].copy()
df['target'] = df_raw['target']  # 0: Malignant, 1: Benign

# Add synthetic categorical columns to demonstrate categorical encoding
risk_levels = ['Low', 'Medium', 'High']
df['risk_level'] = np.random.choice(risk_levels, size=len(df), p=[0.4, 0.4, 0.2])

locations = ['Clinic_A', 'Clinic_B', 'Clinic_C']
df['clinic_location'] = np.random.choice(locations, size=len(df), p=[0.5, 0.3, 0.2])

# Introduce synthetic missing values (NaNs) to demonstrate imputation
mask_radius = np.random.rand(len(df)) < 0.08
df.loc[mask_radius, 'mean radius'] = np.nan

mask_texture = np.random.rand(len(df)) < 0.08
df.loc[mask_texture, 'mean texture'] = np.nan

mask_risk = np.random.rand(len(df)) < 0.05
df.loc[mask_risk, 'risk_level'] = np.nan

print(f"Dataset shape: {df.shape}")
print("\nFirst 5 rows of the modified dataset:")
display(df.head())

## 3. Exploratory Data Analysis (EDA)

Exploratory Data Analysis helps us understand data distributions, missingness patterns, statistical properties, and relationships between features before modifying the data.

Key EDA steps:
- Structural overview (`df.info()`)
- Descriptive statistics (`df.describe()`)
- Missing value analysis
- Visualizations (Correlation heatmap & distribution plots)

In [ ]:
# Dataset structure and non-null counts
print("--- Dataset Information ---")
df.info()

print("\n--- Statistical Summary ---")
display(df.describe().T)

# Missing value inspection
missing_count = df.isnull().sum()
missing_pct = (missing_count / len(df)) * 100
missing_summary = pd.DataFrame({'Missing Count': missing_count, 'Percentage (%)': missing_pct})
print("\n--- Missing Value Summary ---")
display(missing_summary[missing_summary['Missing Count'] > 0])

In [ ]:
# Visualizing correlations and feature distributions
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# 1. Feature Correlation Matrix
num_cols = df.select_dtypes(include=[np.number]).columns
corr_matrix = df[num_cols].corr()
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="coolwarm", ax=axes[0], cbar=True)
axes[0].set_title("Numerical Feature Correlation Matrix")

# 2. Distribution plot of 'mean area' across target classes
sns.histplot(data=df, x="mean area", hue="target", kde=True, element="step", ax=axes[1], palette="Set2")
axes[1].set_title("Distribution of 'mean area' by Target Class")

plt.tight_layout()
plt.show()

## 4. Handling Missing Values (Imputation)

Missing data can cause errors in model training or skew predictions. Standard imputation strategies include:

- **Mean Imputation**: Fills missing values with column mean. Best for normally distributed numerical features without extreme outliers.
- **Median Imputation**: Fills missing values with column median. Robust against skewed distributions and outliers.
- **Mode (Most Frequent) Imputation**: Fills missing values with the most frequent value. Essential for categorical variables.

We use `sklearn.impute.SimpleImputer` for consistent and reproducible imputation.

In [ ]:
# Make a copy of the dataframe for imputation
df_imputed = df.copy()

# Instantiate Imputers
mean_imputer = SimpleImputer(strategy='mean')
median_imputer = SimpleImputer(strategy='median')
mode_imputer = SimpleImputer(strategy='most_frequent')

# Impute 'mean radius' using Mean Imputation
df_imputed['mean radius'] = mean_imputer.fit_transform(df_imputed[['mean radius']])

# Impute 'mean texture' using Median Imputation
df_imputed['mean texture'] = median_imputer.fit_transform(df_imputed[['mean texture']])

# Impute categorical 'risk_level' using Mode Imputation
df_imputed['risk_level'] = mode_imputer.fit_transform(df_imputed[['risk_level']]).ravel()

print("Missing value count after imputation:")
print(df_imputed.isnull().sum())

## 5. Encoding Categorical Variables

Machine learning algorithms operate on numerical mathematical vectors. Categorical variables must be converted to numbers:

1. **Ordinal Encoding (`OrdinalEncoder`)**: Used when categories have a natural ranking or order (e.g., Low < Medium < High).
2. **Label Encoding (`LabelEncoder`)**: Used to encode target labels ($y$) into integer format ($0, 1, 2, \dots$).
3. **One-Hot Encoding (`OneHotEncoder`)**: Creates binary columns for each distinct category. Ideal for non-ordinal (nominal) categories to prevent algorithms from assuming invalid numeric orders.

In [ ]:
df_encoded = df_imputed.copy()

# 1. Ordinal Encoding for ordered categorical feature: 'risk_level'
risk_categories = [['Low', 'Medium', 'High']]
ordinal_enc = OrdinalEncoder(categories=risk_categories)
df_encoded['risk_level_encoded'] = ordinal_enc.fit_transform(df_encoded[['risk_level']])

# 2. Label Encoding for target or high cardinality string representations
label_enc = LabelEncoder()
df_encoded['clinic_location_label'] = label_enc.fit_transform(df_encoded['clinic_location'])

# 3. One-Hot Encoding for nominal feature: 'clinic_location'
ohe = OneHotEncoder(sparse_output=False, drop='first')  # drop='first' avoids multi-collinearity (dummy variable trap)
ohe_array = ohe.fit_transform(df_encoded[['clinic_location']])
ohe_cols = ohe.get_feature_names_out(['clinic_location'])

# Convert one-hot encoded matrix into DataFrame and join
df_ohe = pd.DataFrame(ohe_array, columns=ohe_cols, index=df_encoded.index)
df_encoded = pd.concat([df_encoded, df_ohe], axis=1)

# Drop raw unencoded categorical columns
df_clean = df_encoded.drop(columns=['risk_level', 'clinic_location', 'clinic_location_label'])

print("Features after Categorical Encoding:")
print(df_clean.columns.tolist())
display(df_clean.head())

## 6. Feature Engineering

Feature engineering creates new predictive signals from existing raw features:

- **Domain & Mathematical Features**: Constructing meaningful ratios, products, or log transformations (e.g., perimeter-to-area ratio).
- **Polynomial Features (`PolynomialFeatures`)**: Capturing non-linear relationships and feature interaction terms ($x_1^2, x_2^2, x_1 x_2$).

In [ ]:
df_fe = df_clean.copy()

# 1. Domain Feature Engineering: Perimeter to Area ratio & Radius x Compactness product
df_fe['perimeter_area_ratio'] = df_fe['mean perimeter'] / (df_fe['mean area'] + 1e-5)
df_fe['radius_compactness_product'] = df_fe['mean radius'] * df_fe['mean compactness']

# 2. Polynomial Features for key numerical attributes
poly_cols = ['mean radius', 'mean texture']
poly = PolynomialFeatures(degree=2, include_bias=False)
poly_features = poly.fit_transform(df_fe[poly_cols])
poly_feature_names = poly.get_feature_names_out(poly_cols)

# Convert into DataFrame and combine
poly_df = pd.DataFrame(poly_features, columns=poly_feature_names, index=df_fe.index)
# Drop original duplicates already in df_fe
poly_df = poly_df.drop(columns=poly_cols)

df_fe = pd.concat([df_fe, poly_df], axis=1)

print(f"DataFrame shape after Feature Engineering: {df_fe.shape}")
print("\nSample of newly created features:")
display(df_fe[['perimeter_area_ratio', 'radius_compactness_product'] + list(poly_df.columns)].head())

## 7. Train-Test Split & Data Leakage Prevention

### Critical Concept: Data Leakage
Data leakage occurs when information from the test/validation set leaks into the training pipeline before model training (e.g., computing dataset-wide mean/scaling params before splitting). To prevent leakage:

1. Perform `train_test_split` **BEFORE** feature scaling or model training.
2. Call `.fit()` or `.fit_transform()` **ONLY** on the training set (`X_train`).
3. Call `.transform()` on the test set (`X_test`).

In [ ]:
# Separate predictor features (X) and target variable (y)
X = df_fe.drop(columns=['target'])
y = df_fe['target']

# Stratified Train-Test Split (80% Train, 20% Test)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print(f"X_train dimensions: {X_train.shape}")
print(f"X_test dimensions:  {X_test.shape}")
print("\nTarget proportion in Training set:")
print(y_train.value_counts(normalize=True))

## 8. Feature Scaling

Distance-based algorithms (k-NN, SVM, Logistic Regression, Gradient Descent) require features to be on comparable numeric scales.

We demonstrate three main scalers:

1. **StandardScaler**: Centers data around mean $\mu=0$ with standard deviation $\sigma=1$.
2. **MinMaxScaler**: Rescales data linearly into range $[0, 1]$.
3. **RobustScaler**: Uses Median and Interquartile Range (IQR); resistant to extreme outliers.

In [ ]:
# Initialize Scaler instances
std_scaler = StandardScaler()
minmax_scaler = MinMaxScaler()
robust_scaler = RobustScaler()

# Fit ONLY on X_train, then transform both X_train and X_test
X_train_std = std_scaler.fit_transform(X_train)
X_test_std = std_scaler.transform(X_test)

X_train_minmax = minmax_scaler.fit_transform(X_train)
X_test_minmax = minmax_scaler.transform(X_test)

X_train_robust = robust_scaler.fit_transform(X_train)
X_test_robust = robust_scaler.transform(X_test)

# Visual comparison of scaling methods on 'mean area'
fig, axes = plt.subplots(1, 4, figsize=(20, 4.5))

col_idx = X_train.columns.get_loc('mean area')

sns.kdeplot(X_train.iloc[:, col_idx], ax=axes[0], color='black', fill=True)
axes[0].set_title("Original Data ('mean area')")

sns.kdeplot(X_train_std[:, col_idx], ax=axes[1], color='blue', fill=True)
axes[1].set_title("StandardScaler")

sns.kdeplot(X_train_minmax[:, col_idx], ax=axes[2], color='green', fill=True)
axes[2].set_title("MinMaxScaler")

sns.kdeplot(X_train_robust[:, col_idx], ax=axes[3], color='purple', fill=True)
axes[3].set_title("RobustScaler")

plt.tight_layout()
plt.show()

## 9. Feature Selection Techniques

Feature selection reduces dimensionality, decreases memory usage, speeds up training, and helps prevent overfitting.

Three main categories of Feature Selection:
1. **Filter Methods (`SelectKBest`)**: Ranks features based on statistical tests (e.g., ANOVA F-test, Chi-square).
2. **Wrapper Methods (`RFE`)**: Recursively trains models and removes features with lowest predictive power.
3. **Embedded Methods (`Feature Importance`)**: Uses built-in tree model feature importance metrics (e.g., Random Forest Gini impurity reduction).

In [ ]:
# 1. Filter Method: SelectKBest (ANOVA F-test)
kbest_selector = SelectKBest(score_func=f_classif, k=5)
X_train_kbest = kbest_selector.fit_transform(X_train_std, y_train)

selected_kbest = X_train.columns[kbest_selector.get_support()].tolist()
print("Top 5 Features selected by SelectKBest (ANOVA F-test):")
print(selected_kbest)

# 2. Wrapper Method: Recursive Feature Elimination (RFE)
rf_base = RandomForestClassifier(n_estimators=50, random_state=42)
rfe_selector = RFE(estimator=rf_base, n_features_to_select=5, step=1)
rfe_selector.fit(X_train_std, y_train)

selected_rfe = X_train.columns[rfe_selector.support_].tolist()
print("\nTop 5 Features selected by RFE (Random Forest Wrapper):")
print(selected_rfe)

# 3. Embedded Method: Random Forest Feature Importances
rf_full = RandomForestClassifier(n_estimators=100, random_state=42)
rf_full.fit(X_train_std, y_train)

importances = pd.Series(rf_full.feature_importances_, index=X_train.columns).sort_values(ascending=False)
top10 = importances.head(10)

# Visualization of Feature Importances
plt.figure(figsize=(10, 5))
sns.barplot(x=top10.values, y=top10.index, hue=top10.index, palette="viridis", legend=False)
plt.title("Top 10 Feature Importances via Random Forest Gini Score")
plt.xlabel("Importance Score")
plt.ylabel("Feature")
plt.show()

## 10. Summary & Conclusion

### Preprocessing Summary Workflow:

1. **Library Imports**: Loaded pandas, numpy, scikit-learn, matplotlib, seaborn.
2. **Dataset Setup**: Utilized Breast Cancer dataset with added categorical and missing features.
3. **EDA**: Inspected summary statistics, missing data counts, correlation heatmaps, and feature distributions.
4. **Missing Value Imputation**: Replaced NaNs using `SimpleImputer` (Mean, Median, Mode).
5. **Categorical Encoding**: Applied `OrdinalEncoder`, `LabelEncoder`, and `OneHotEncoder` appropriately.
6. **Feature Engineering**: Created domain ratio interactions and `PolynomialFeatures` terms.
7. **Train-Test Split**: Divided data into train/test sets to strictly prevent Data Leakage.
8. **Feature Scaling**: Implemented `StandardScaler`, `MinMaxScaler`, and `RobustScaler`.
9. **Feature Selection**: Evaluated key features using `SelectKBest`, `RFE`, and `RandomForest` importances.

---